# QAOA on IBM Quantum for MTDA

Solve MTDA QUBO using QAOA (Quantum Approximate Optimization Algorithm):
- Standard QAOA via Qiskit Optimization
- FPC-QAOA with fixed parameter count (arXiv:2512.21181)
- Comparison with D-Wave annealing and classical solvers

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from quantum_common.visualization.styles import apply_publication_style, QUANTUM_COLORS

apply_publication_style()

from quantum_mht.formulation.mtda_qubo_builder import MTDAQuboBuilder
from quantum_mht.solvers.solver_factory import create_solver

In [ ]:
# Small problem for QAOA (3 targets to keep qubit count manageable)
rng = np.random.default_rng(42)
n_targets = 3
predicted = rng.uniform(20, 80, size=(n_targets, 2))
measurements = np.vstack([
    predicted + rng.normal(0, 2, size=(n_targets, 2)),
    rng.uniform(0, 100, size=(2, 2)),  # 2 clutter
])
covs = np.array([np.eye(2) * 4.0 for _ in range(n_targets)])

builder = MTDAQuboBuilder()
cost_matrix = builder.cost_builder.build(predicted, measurements, covs)
qubo = builder.build_from_cost_matrix(cost_matrix)
print(f"QUBO: {qubo.num_variables} variables (qubits for QAOA)")

In [ ]:
# Solve with QAOA
try:
    qaoa = create_solver('qaoa', reps=2, use_simulator=True)
    result_qaoa = qaoa.solve(qubo)
    print(f"QAOA(p=2): obj={result_qaoa.objective_value:.2f}, time={result_qaoa.solve_time_s:.4f}s")
    print(f"  Assignments: {result_qaoa.assignments}")
    print(f"  Feasible: {result_qaoa.is_feasible}")
except ImportError:
    print("qiskit-optimization not installed - install with: pip install qiskit-optimization")
except Exception as e:
    print(f"QAOA failed: {e}")

In [ ]:
# Compare all available solvers
import time

solver_results = {}

# Hungarian (optimal)
hung = create_solver('hungarian')
solver_results['Hungarian'] = hung.solve(qubo)

# GNN (greedy)
gnn = create_solver('gnn')
solver_results['GNN'] = gnn.solve(qubo)

# Annealing simulator
try:
    anneal = create_solver('annealing', use_simulator=True, num_reads=200)
    solver_results['Annealing'] = anneal.solve(qubo)
except Exception:
    pass

# QAOA
try:
    qaoa_solver = create_solver('qaoa', reps=1)
    solver_results['QAOA'] = qaoa_solver.solve(qubo)
except Exception:
    pass

print(f"{'Solver':>15} {'Objective':>10} {'Time(s)':>10} {'Feasible':>8} {'Assigns':>8}")
print('-' * 56)
for name, r in solver_results.items():
    print(f"{name:>15} {r.objective_value:>10.2f} {r.solve_time_s:>10.4f} {str(r.is_feasible):>8} {len(r.assignments):>8}")